# Freeze baselines

Esegue una volta la GNN originale e il B-score logit, quindi salva le
predizioni test match-per-match. Rieseguire solo se cambiano scope, split,
seed o protocollo di training.

In [ ]:
import os
from dataclasses import asdict
from pathlib import Path

START_DIRECTORY = Path.cwd().resolve()
if (START_DIRECTORY / "edge_feature_ablation").is_dir():
    PROJECT_ROOT = START_DIRECTORY
elif START_DIRECTORY.name == "experiments":
    PROJECT_ROOT = START_DIRECTORY.parent.parent
else:
    PROJECT_ROOT = START_DIRECTORY.parent

EDGE_DIRECTORY = PROJECT_ROOT / "edge_feature_ablation"
RESULTS_ROOT = PROJECT_ROOT / "results" / "frozen_predictions"

In [ ]:
TOURNAMENT_SCOPE = os.environ.get(
    "TENNIS_RUN_SCOPE", "slams_masters"
)  # or "slams"
EXPERIMENT_NAME = "current_gnn"
EDGE_ABLATION = "current"

In [ ]:
os.environ["TENNIS_MODEL_EXPERIMENT"] = EXPERIMENT_NAME
if EDGE_ABLATION is None:
    os.environ.pop("TENNIS_EDGE_ABLATION", None)
else:
    os.environ["TENNIS_EDGE_ABLATION"] = EDGE_ABLATION
os.environ["TENNIS_TOURNAMENT_SCOPE"] = TOURNAMENT_SCOPE

previous_directory = Path.cwd()
os.chdir(EDGE_DIRECTORY)
try:
    get_ipython().run_line_magic("run", "GNN_edge_feature_ablation.ipynb")
finally:
    os.chdir(previous_directory)

In [ ]:
import sys
sys.path.insert(0, str(EDGE_DIRECTORY))
from experiment_tracking import ArtifactManifest, save_prediction_artifact

OUTPUT_DIRECTORY = (
    RESULTS_ROOT / TOURNAMENT_SCOPE / f"seed_{SEED}"
)

def make_manifest(name, family, config):
    return ArtifactManifest(
        experiment=name,
        model_family=family,
        tournament_scope=TOURNAMENT_SCOPE,
        seed=SEED,
        train_end=TRAIN_END,
        validation_end=VAL_END,
        test_end=int(future_matches["tourney_date"].dt.year.max()),
        update_phases=tuple(sorted(UPDATE_PHASES)),
        config=config,
    )

In [ ]:
test_gnn = predictions_df[predictions_df["phase"] == "test"].copy()
gnn_paths = save_prediction_artifact(
    test_gnn,
    OUTPUT_DIRECTORY,
    make_manifest(
        "current_gnn",
        "GINE",
        {"model": asdict(MODEL_CONFIG), "edge": asdict(EDGE_CONFIG)},
    ),
)

test_logit = test_predictions.copy()
logit_paths = save_prediction_artifact(
    test_logit,
    OUTPUT_DIRECTORY,
    make_manifest(
        "bscore_logit",
        "LogisticRegression",
        {"features": ["x_bscore_diff_scaled"]},
    ),
    probability_column="bscore_prob",
)

print("Frozen GNN:", gnn_paths)
print("Frozen B-score logit:", logit_paths)